In [ ]:
import json
from pathlib import Path
import pandas as pd

# Auto-discover all model directories
base_dir = Path(".")
rows = []

for model_dir in sorted(base_dir.iterdir()):
    if not model_dir.is_dir() or model_dir.name.startswith("."):
        continue
    
    for f in sorted(model_dir.glob("*.json")):
        with open(f) as fp:
            data = json.load(fp)
        
        k = data.get("few_shot_k", 0)
        model = data.get("model", model_dir.name)
        
        for dataset, metrics in data["datasets"].items():
            rows.append({
                "model": model,
                "k": k,
                "total_examples": 2 * k,
                "dataset": dataset,
                "f1_original": metrics["macro_f1_original"],
                "f1_content_only": metrics["macro_f1_content_only"],
                "f1_shuffle": metrics["macro_f1_shuffle"],
                "delta_shuffle": metrics["delta_shuffle"],
            })

df = pd.DataFrame(rows).sort_values(["model", "k"])
print(f"Loaded {len(df)} results from {df['model'].nunique()} model(s)")
df

In [ ]:
# Summary: F1 by model and k value
summary = df.groupby(["model", "k"])[["f1_original", "f1_shuffle", "delta_shuffle"]].mean().round(4)
summary["total_examples"] = [k * 2 for _, k in summary.index]
summary = summary[["total_examples", "f1_original", "f1_shuffle", "delta_shuffle"]]
print("=== F1 by model and k (few-shot examples) ===")
print(summary.to_string())

# Per-model improvement
print("\n=== Improvement from k=0 to max k ===")
for model in df["model"].unique():
    model_df = df[df["model"] == model]
    k_min, k_max = model_df["k"].min(), model_df["k"].max()
    f1_min = model_df[model_df["k"] == k_min]["f1_original"].mean()
    f1_max = model_df[model_df["k"] == k_max]["f1_original"].mean()
    print(f"{model}: k={k_min}→k={k_max}: {f1_min:.4f} → {f1_max:.4f} ({f1_max - f1_min:+.4f})")